In [1]:
import os

path = "/Users/henrytan/Library/CloudStorage/OneDrive-SunwayGroup/Noelle Tan Huey Wen's files - Theme Park/data.parquet"

if not os.path.exists(path):
    print("Path does not exist!")
elif os.path.isdir(path):
    print("This is a DIRECTORY, not a single file.")
else:
    print(f"File size: {os.path.getsize(path) / (1024**3):.2f} GB")

File size: 4.10 GB


In [3]:
import duckdb

# 1. Your original path
path = "/Users/henrytan/Library/CloudStorage/OneDrive-SunwayGroup/Noelle Tan Huey Wen's files - Theme Park/data.parquet"

# 2. ESCAPE THE QUOTE for SQL: This changes "Wen's" to "Wen''s" inside the SQL string
sql_path = path.replace("'", "''")

con = duckdb.connect("theme_park_data.db")

print("Starting import... this should work now!")

# 3. Use the escaped 'sql_path' inside the query
con.execute(f"CREATE OR REPLACE TABLE theme_park_database AS SELECT * FROM read_parquet('{sql_path}');")

print("Import complete!")

# 4. Verify
row_count = con.execute("SELECT count(*) FROM theme_park_database").fetchone()[0]
print(f"✅ Total rows imported: {row_count:,}")

con.close()

Starting import... this should work now!


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Import complete!
✅ Total rows imported: 44,037,386


In [4]:
import duckdb

# Connect to the database you just created
con = duckdb.connect("theme_park_data.db")

# 1. See the table structure (Column names and Types)
print("--- Table Schema ---")
con.sql("DESCRIBE theme_park_database").show()

# 2. See the first 5 rows (The SQL way)
print("\n--- Data Preview (First 5 Rows) ---")
con.sql("SELECT * FROM theme_park_database LIMIT 5").show()

con.close()

--- Table Schema ---
┌───────────────────┬───────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │  column_type  │  null   │   key   │ default │  extra  │
│      varchar      │    varchar    │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼───────────────┼─────────┼─────────┼─────────┼─────────┤
│ NodeNo            │ INTEGER       │ YES     │ NULL    │ NULL    │ NULL    │
│ FiscalDate        │ TIMESTAMP     │ YES     │ NULL    │ NULL    │ NULL    │
│ TranDate          │ TIMESTAMP     │ YES     │ NULL    │ NULL    │ NULL    │
│ CompanyID         │ INTEGER       │ YES     │ NULL    │ NULL    │ NULL    │
│ Abbr              │ VARCHAR       │ YES     │ NULL    │ NULL    │ NULL    │
│ Name              │ VARCHAR       │ YES     │ NULL    │ NULL    │ NULL    │
│ Environment       │ VARCHAR       │ YES     │ NULL    │ NULL    │ NULL    │
│ AccountID         │ VARCHAR       │ YES     │ NULL    │ NULL    │ NULL    │
│ Qty               │ INTEGER       │ YES  

In [5]:
import duckdb

con = duckdb.connect("theme_park_data.db")

# Example: Pull only a specific subset into a Pandas DataFrame
# Let's say you only want data from a specific year or category
df = con.execute("""
    SELECT * FROM theme_park_database 
    WHERE Qty >= 1000
    LIMIT 1000
""").df()

print(f"Dataframe shape: {df.shape}")
con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Dataframe shape: (62, 122)


In [6]:
df.head()

,NodeNo,FiscalDate,TranDate,CompanyID,Abbr,Name,Environment,AccountID,Qty,Amount,...,OriginalAmount,JnlDetailID,CatCode17Desc1,Last_In_Unit_Cost,Average_Unit_Cost,CatCode18Desc1,TranNo,ReceiptNo,SurveyID,Race
0,91,2022-11-20 16:00:00,2022-11-21 03:48:56,1,None,None,1,000110210001,4290,48562.80,...,48562.80,167185164,NaN,NaN,NaN,NaN,73934,21956,<NA>,None
1,232,2023-09-19 16:00:00,2023-09-20 08:17:20,1,None,None,2,000110231001,1051,69408.04,...,69408.04,109172884,Revenue,NaN,NaN,Group Function-Entrance,23115,8107,<NA>,None
2,232,2023-11-20 16:00:00,2023-11-21 08:53:12,1,None,None,2,000110231001,1266,20306.64,...,20306.64,111181436,Revenue,NaN,NaN,Group Function-Entrance,30086,10972,<NA>,None
3,232,2023-12-20 16:00:00,2023-12-21 03:42:34,1,None,None,2,000110231001,1500,14145.00,...,14145.00,112295983,Revenue,NaN,NaN,Group Function-Entrance,33765,12520,<NA>,None
4,91,2022-10-06 16:00:00,2022-10-07 09:03:43,1,None,None,1,000110210001,3069,63681.75,...,63681.75,165113198,NaN,NaN,NaN,NaN,72691,21686,<NA>,None


In [11]:
df.describe()

,NodeNo,FiscalDate,TranDate,CompanyID,Qty,Amount,OrderLineID,UserID,GLCOde,Category,...,Discount_ValidFrom,AgencyNo,Percentage,OriginalAmount,JnlDetailID,Last_In_Unit_Cost,Average_Unit_Cost,TranNo,ReceiptNo,SurveyID
count,62.000000,62,62,60.0,62.000000,62.000000,60.0,60.0,62.0,62.000000,...,0,60.0,60.0,62.000000,60.0,1.0,1.000,60.0,60.0,0.0
mean,174.467742,2023-07-19 16:00:00,2023-07-20 07:07:00.112903,1.45,2665.000000,31091.715323,723645.0,2122.866667,102.0,252.403226,...,NaT,13.616667,1.0,31091.715323,142289624.283333,0.0,0.567,67697.016667,26727.15,<NA>
min,55.000000,2017-01-30 16:00:00,2017-01-30 16:00:00,1.0,1014.000000,0.000000,0.0,52.0,102.0,9.000000,...,NaT,7.0,1.0,0.000000,94486606.0,0.0,0.567,6539.0,934.0,<NA>
25%,91.000000,2022-11-23 22:00:00,2022-11-24 14:47:09,1.0,1266.000000,7639.250000,660071.75,52.0,102.0,100.000000,...,NaT,7.0,1.0,7639.250000,109630209.25,0.0,0.567,28489.75,8999.25,<NA>
50%,231.500000,2023-09-25 16:00:00,2023-09-26 07:11:03.500000,1.0,1486.500000,20811.900000,783561.0,52.0,102.0,310.000000,...,NaT,7.0,1.0,20811.900000,123966176.5,0.0,0.567,71570.0,21833.5,<NA>
75%,232.000000,2024-08-10 10:00:00,2024-08-11 02:59:00.250000,1.0,2700.000000,35667.000000,813375.75,3037.0,102.0,310.000000,...,NaT,24.0,1.0,35667.000000,167345492.5,0.0,0.567,77378.0,23236.5,<NA>
max,235.000000,2025-07-11 16:00:00,2025-07-12 08:47:51,10.0,20000.000000,188600.000000,1363137.0,9790.0,102.0,901.000000,...,NaT,24.0,1.0,188600.000000,243488769.0,0.0,0.567,437469.0,374499.0,<NA>
std,69.821956,NaN,NaN,1.978058,3420.619396,37644.855856,302602.441592,3109.0061,0.0,193.176761,...,NaN,8.155463,0.0,37644.855856,41502904.078991,NaN,NaN,56990.827485,48501.364954,<NA>
